In [ ]:
import sys
from pathlib import Path
print(Path.cwd())
from vorstellungsgesprach.utils import load_data, add_language_metadata,remove_duplicate_jobs
from vorstellungsgesprach import conf
%cd /Users/eli/vorstellungsgesprach
from dotenv import load_dotenv
from google import genai
import os
load_dotenv()

In [ ]:
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [ ]:
jobs = load_data("data/raw/job.json")
jobs = [
    {
        "id": job["id"],
        "description": job.get("description", ""),
        "title": job.get("title", ""),
        "company": job.get("companyName", "")
    }
    for job in jobs
]
print(len(jobs))


In [ ]:
jobs[:2]

In [ ]:
jobs_with_language = add_language_metadata(jobs)
jobs_with_language

In [ ]:
german_jobs = [
    job
    for job in jobs_with_language
    if job["detected_language"] == "german"
    and job["language_confidence"] >= 0.80
]

In [ ]:
print(len(german_jobs)) 

In [ ]:
german_jobs[:4]

In [ ]:
company_jobs = [
    job for job in jobs
    if job.get("company") == "Deloitte"
]

In [ ]:
company_jobs

In [ ]:
unique_jobs = remove_duplicate_jobs(german_jobs)

print(f"Before: {len(german_jobs)}")
print(f"After: {len(unique_jobs)}")
print(f"Duplicates removed: {len(german_jobs) - len(unique_jobs)}")

In [ ]:
unique_jobs

In [ ]:
unique_titles = {
    unique_job.get("title", "")
    for unique_job in unique_jobs
}

for title in sorted(unique_titles):
    print(title)

In [ ]:
lengths = [len(job["description"]) for job in unique_jobs]
print(f"Média de caracteres: {sum(lengths) / len(lengths):.0f}")
print(f"Máximo: {max(lengths)}")
print(f"Mínimo: {min(lengths)}")

In [ ]:
def chunk_description(text: str, max_chars: int = 4000, overlap: int = 200) -> list[str]:
    """Quebra a descrição em pedaços se for muito longa; senão, retorna como está."""
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap  # overlap evita cortar contexto no meio de uma frase importante

    return chunks


def build_documents(jobs: list[dict]) -> list[dict]:
    """Transforma cada vaga em um ou mais documentos, dependendo do tamanho."""
    documents = []
    for job in jobs:
        description_chunks = chunk_description(job["description"])

        for i, chunk in enumerate(description_chunks):
            text = f"Titel: {job['title']}\nUnternehmen: {job['company']}\n\n{chunk}"
            documents.append({
                "id": f"{job['id']}_chunk{i}" if len(description_chunks) > 1 else job["id"],
                "text": text,
                "metadata": {
                    "job_id": job["id"],
                    "title": job["title"],
                    "company": job["company"],
                    "chunk_index": i,
                    "total_chunks": len(description_chunks),
                },
            })
    return documents


documents = build_documents(unique_jobs)
print(f"Total de vagas: {len(unique_jobs)}")
print(f"Total de documentos (após chunking): {len(documents)}")

In [ ]:
print("Chave carregada:", bool(conf.GEMINI_API_KEY))
print("Modelo de embedding:", conf.GEMINI_EMBEDDING_MODEL)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Xenova/all-MiniLM-L6-v2",
    local_dir="models/Xenova/all-MiniLM-L6-v2",
    allow_patterns=["tokenizer.json", "onnx/model.onnx", "config.json"],
)

In [ ]:
import chromadb
from vorstellungsgesprach import embeddings

texts = [doc["text"] for doc in documents]
vectors = embeddings.embed_texts(texts)

print(f"Embeddings gerados: {len(vectors)}")
print(f"Dimensão de cada vetor: {len(vectors[0])}")

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="vagas_ti")

collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=vectors,
    documents=[doc["text"] for doc in documents],
    metadatas=[doc["metadata"] for doc in documents],
)

print(f"Documentos indexados: {collection.count()}")

In [ ]:
query = "Welche Stellen erfordern Erfahrung mit Python?"
query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"\n--- {meta['title']} @ {meta['company']} (distância: {distance:.3f}) ---")
    print(doc[:200])

In [ ]:
perguntas_teste = [
    "Welche Jobs sind für Berufseinsteiger geeignet?",
    "Gibt es Stellen im Bereich Machine Learning?",
    "Welche Unternehmen bieten Home-Office an?",
]

for pergunta in perguntas_teste:
    print(f"\n{'='*60}\nPergunta: {pergunta}\n{'='*60}")
    qv = embeddings.embed_query(pergunta)
    res = collection.query(query_embeddings=[qv], n_results=2)
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        print(f"→ {meta['title']} @ {meta['company']}")

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

for collection in client.list_collections():
    print(collection.name)
collection = client.get_collection(name="vagas_ti")
print(collection.count())

In [ ]:
import chromadb
from google import genai

from vorstellungsgesprach import conf, rag
from vorstellungsgesprach.models import list_available_chat_models


CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "vagas_ti"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

question = "Welche Stellen erfordern Erfahrung mit Python?"

gemini_client = genai.Client(api_key=conf.GEMINI_API_KEY)
available_models = list_available_chat_models(gemini_client)

results = []

for model_name in available_models:
    try:
        result = rag.answer(
            collection=collection,
            query=question,
            model=model_name,
            n_results=20,
        )

        results.append(result)

        print(f"\n{'=' * 80}")
        print(f"Modelo: {result['model']}")
        print(f"{'=' * 80}")
        print(result["answer"])

    except Exception:
        continue

if results:
    print("\nFontes recuperadas:")

    for index, source in enumerate(results[0]["sources"], start=1):
        metadata = source["metadata"]

        print(
            f"{index}. {metadata.get('title', 'Sem título')} "
            f"@ {metadata.get('company', 'Sem empresa')} "
            f"(distância: {source['distance']:.3f})"
        )
else:
    print("Nenhum modelo produziu uma resposta.")

In [ ]:
stored_data = collection.get(
    include=["documents", "metadatas"],
)

python_jobs = []

for document, metadata in zip(
    stored_data["documents"],
    stored_data["metadatas"],
):
    if "python" in document.casefold():
        python_jobs.append(
            {
                "title": metadata.get("title", ""),
                "company": metadata.get("company", ""),
            }
        )

print(f"Vagas que mencionam Python: {len(python_jobs)}")

for job in python_jobs:
    print(f"{job['title']} @ {job['company']}")

In [ ]:
for source in results[0]["sources"]:
    text = source["text"]
    position = text.casefold().find("Python")

    print(f"\n{source['metadata']['title']}")

    if position >= 0:
        start = max(0, position - 200)
        end = min(len(text), position + 300)
        print(text[start:end])
    else:
        print("Python não aparece neste documento.")

In [ ]:
from pathlib import Path

import chromadb

from vorstellungsgesprach import embeddings


CHROMA_PATH = Path("./chroma_db").resolve()
COLLECTION_NAME = "vagas_ti"

print(f"ChromaDB usado: {CHROMA_PATH}")

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_collection(name=COLLECTION_NAME)

print(f"Documentos: {collection.count()}")

question = "Welche Stellen erfordern Erfahrung mit Python?"
query_vector = embeddings.embed_query(question)

query_results = collection.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

for document_id, document, metadata, distance in zip(
    query_results["ids"][0],
    query_results["documents"][0],
    query_results["metadatas"][0],
    query_results["distances"][0],
):
    contains_python = "python" in document.casefold()

    print(
        f"{document_id} | "
        f"{metadata.get('title')} | "
        f"Python: {contains_python} | "
        f"Distância: {distance:.3f}"
    )

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer


CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "vagas_ti_multilingual"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu",
)

texts = [document["text"] for document in documents]

vectors = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

print(f"Vetores gerados: {len(vectors)}")
print(f"Dimensão: {len(vectors[0])}")

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection_multilingual = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection_multilingual.upsert(
    ids=[document["id"] for document in documents],
    embeddings=vectors,
    documents=[document["text"] for document in documents],
    metadatas=[document["metadata"] for document in documents],
)

print(f"Documentos indexados: {collection_multilingual.count()}")

In [ ]:
question = "Welche Stellen erfordern Erfahrung mit Python?"

query_vector = model.encode(
    question,
    normalize_embeddings=True,
).tolist()

results = collection_multilingual.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(
        f"{metadata.get('title')} @ "
        f"{metadata.get('company')} | "
        f"Python: {'python' in document.casefold()} | "
        f"Distância: {distance:.3f}"
    )

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu",
)

test_vectors = model.encode(
    [
        "Welche Stellen erfordern Erfahrung mit Python?",
        "Erfahrung mit Python und Machine Learning ist erforderlich.",
    ],
    normalize_embeddings=True,
)

print(test_vectors.shape)

In [ ]:
import chromadb


CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "vagas_ti_multilingual"

texts = [document["text"] for document in documents]

vectors = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection_multilingual = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection_multilingual.upsert(
    ids=[document["id"] for document in documents],
    embeddings=vectors,
    documents=texts,
    metadatas=[document["metadata"] for document in documents],
)

print(f"Documentos indexados: {collection_multilingual.count()}")
print(f"Dimensão: {len(vectors[0])}")

In [ ]:
question = "Welche Stellen erfordern Erfahrung mit Python?"

query_vector = model.encode(
    question,
    normalize_embeddings=True,
).tolist()

results = collection_multilingual.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(
        f"{metadata.get('title')} @ "
        f"{metadata.get('company')} | "
        f"Python: {'python' in document.casefold()} | "
        f"Distância: {distance:.3f}"
    )

In [ ]:
question = "Python Kenntnisse Erfahrung Anforderungen"

query_vector = model.encode(
    question,
    normalize_embeddings=True,
).tolist()

results = collection_multilingual.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(
        f"{metadata.get('title')} @ "
        f"{metadata.get('company')} | "
        f"Python: {'python' in document.casefold()} | "
        f"Distância: {distance:.3f}"
    )

In [ ]:
def hybrid_search(
    collection,
    model,
    query: str,
    keywords: list[str],
    n_results: int = 10,
) -> list[dict]:
    """Combina busca semântica com termos técnicos exatos."""
    query_vector = model.encode(
        query,
        normalize_embeddings=True,
    ).tolist()

    total_documents = collection.count()

    semantic_results = collection.query(
        query_embeddings=[query_vector],
        n_results=total_documents,
        include=["documents", "metadatas", "distances"],
    )

    hits = []

    for document_id, document, metadata, distance in zip(
        semantic_results["ids"][0],
        semantic_results["documents"][0],
        semantic_results["metadatas"][0],
        semantic_results["distances"][0],
    ):
        normalized_document = document.casefold()

        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword.casefold() in normalized_document
        ]

        hits.append(
            {
                "id": document_id,
                "text": document,
                "metadata": metadata,
                "distance": distance,
                "matched_keywords": matched_keywords,
            }
        )

    hits.sort(
        key=lambda hit: (
            not bool(hit["matched_keywords"]),
            hit["distance"],
        )
    )

    return hits[:n_results]

In [ ]:
question = "Welche Stellen erfordern Erfahrung mit Python?"

results = hybrid_search(
    collection=collection_multilingual,
    model=model,
    query=question,
    keywords=["python"],
    n_results=10,
)

for result in results:
    metadata = result["metadata"]

    print(
        f"{metadata.get('title')} @ "
        f"{metadata.get('company')} | "
        f"Python: {bool(result['matched_keywords'])} | "
        f"Distância: {result['distance']:.3f}"
    )

In [2]:
import language_tool_python

tool = language_tool_python.LanguageTool('de-DE')

texto = "Erfahrung im Machine Learning ist erforderlich."
matches = tool.check(texto)

for match in matches:
    print(f"Erro: {match.message}")
    print(f"Sugestão: {match.replacements}")
    print(f"Contexto: {match.context}\n")

In [4]:
texto_com_erro = "Der Mann hat gestern nach Hause gegangt und er haben Hunger."
matches = tool.check(texto_com_erro)

print(f"Erros encontrados: {len(matches)}\n")
for match in matches:
    print(f"Erro: {match.message}")
    print(f"Sugestão: {match.replacements}")
    print(f"Contexto: {match.context}\n")

Erros encontrados: 1

Erro: Möglicher Tippfehler gefunden.
Sugestão: ['gelangt', 'gehängt', 'gekannt', 'geknackt', 'gedankt', 'getankt', 'gebangt', 'gegongt', 'gekackt', 'gekalkt', 'gerankt', 'gewankt', 'gezankt', 'bekannt', 'gelang', 'genannt', 'gekauft', 'gelingt', 'gegangen', 'gesagt']
Contexto: Der Mann hat gestern nach Hause gegangt und er haben Hunger.



In [5]:
from vorstellungsgesprach import grammar

resposta_usuario = "Ich habe drei Jahre Erfahrung mit Python gehabt und ich haben auch mit Machine Learning gearbeitet."

issues = grammar.check_text(resposta_usuario)
print(grammar.format_issues_for_display(issues))

print("\nTexto corrigido automaticamente:")
print(grammar.correct_text(resposta_usuario))

Keine Grammatikfehler gefunden. ✅

Texto corrigido automaticamente:
Ich habe drei Jahre Erfahrung mit Python gehabt und ich haben auch mit Machine Learning gearbeitet.
